# Which change should the peanut-sheller designer make first?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmudrc/design-research-workshop/blob/workshop-three-agent-demos/notebooks/peanut_sheller_propose_critic.ipynb)

A small grower wants a hand-operated tabletop sheller. A proposer specifies the mechanisms. A critic checks the design and assigns changes. Give the proposer one change per round and see which features make it into the next draft.

Run the numbered sections in order. A CPU runtime is enough. The workshop PyBot uses local Python rules and runs without an API key.

## 1. Set up
Run the install cell once in Colab. If you installed the workshop package locally, start in the next section.

In [ ]:
# @title Install the workshop package { display-mode: "form" }
%pip -q install "design-research-workshop @ git+https://github.com/cmudrc/design-research-workshop.git@workshop-three-agent-demos"

## 2. Describe the starting design

Pick a feeder, cracker, and separator. In the text fields, describe features the concept already has. Leave a field blank to use its fixed starting provision. The example side panel gives the critic an unfamiliar cleaning method to review.

For a second run, try `Our prototype has an adjustable roller gap.` with the roller cracker, or `sieve and tray lift out` with the hand-shaken sieve. How do those details affect the first review?

In [1]:
# @title Starting concept { display-mode: "form" }
import json

from design_research.problems import get_problem
from design_research_workshop import workshop_client
from design_research_workshop.peanut import (
    BRIEF, PROBLEM_ID, REVIEW_CRITERIA,
    ShellerConcept, sheller_pattern, sheller_task,
)

feed = "Gravity hopper"  # @param ["Hand-fed tray", "Gravity hopper", "Indexed feed wheel"]
crack = "Hand-cranked rollers"  # @param ["Hinged lever press", "Hand-cranked rollers", "Sliding rub plates"]
separate = "Hand-shaken sieve"  # @param ["Tilted screen", "Hand-shaken sieve", "Hand-cranked air separator"]
kernel_provision = ""  # @param {type:"string"}
cleaning_provision = "A hinged side panel exposes the separator and its collection tray."  # @param {type:"string"}
feed_provision = ""  # @param {type:"string"}

notes = {
    "kernel_protection": kernel_provision,
    "cleaning_access": cleaning_provision,
    "feed_control": feed_provision,
}
concept = ShellerConcept(feed, crack, separate, **{k: v.strip() for k, v in notes.items() if v.strip()})

print(BRIEF)
print(f"Adapted from the problem library's “{get_problem(PROBLEM_ID).metadata.title}” brief.")
print(f"\nStarting concept: {feed} → {crack} → {separate}")
print("Cracking clearance:", concept.kernel_protection)
print("Cleaning access:", concept.cleaning_access)
print("Feed control:", concept.feed_control)

Design a non-electric tabletop peanut sheller for a small grower. Aim to keep kernels intact, use readily available materials, and make food-contact parts easy to clean.
Adapted from the problem library's “Device to shell peanuts” brief.

Starting concept: Gravity hopper → Hand-cranked rollers → Hand-shaken sieve
Cracking clearance: Fixed clearance with no size adjustment during use.
Cleaning access: A hinged side panel exposes the separator and its collection tray.
Feed control: The feeder has no specified setting or release interval.


## 3. Set the critic's priorities and run the loop

The critic looks for specified design features. It can recognize an adjustable cracking limit, access to the separator and tray for cleaning, and a way to meter incoming pods. Choose the criteria and their order. Predict which provision you will see in round two.

In [2]:
# @title Review assignment { display-mode: "form" }
review_order = "kernel protection → cleaning access"  # @param ["kernel protection → cleaning access", "cleaning access → kernel protection", "feed control → cleaning access → kernel protection", "cleaning access"]
changes_per_revision = 1  # @param {type:"slider", min:1, max:3, step:1}
max_iterations = 3  # @param {type:"slider", min:1, max:4, step:1}
criteria = review_order.split(" → ")
task = sheller_task(concept, criteria=criteria, changes_per_revision=changes_per_revision)

with workshop_client() as client:
    result = sheller_pattern(client, max_iterations=max_iterations).run(task)

if result.terminated_reason not in {"approved", "max_iterations_reached"}:
    raise RuntimeError(f"The review stopped early: {result.terminated_reason}")

print(f"Review order: {review_order}")
print(f"Rounds used: {result.iterations}/{max_iterations}")
print(f"Stopping reason: {result.terminated_reason}")

Review order: kernel protection → cleaning access
Rounds used: 3/3
Stopping reason: approved


## 4. Inspect the revision trace

**Accepted** means the description matches a known provision for the selected mechanism. **Missing** means the draft specifies a fixed or absent feature. **Unrecognized** means PyBot needs another way to evaluate the description. In this exercise it proposes a catalog alternative. Decide whether you would keep the original side panel for a prototype test.

In [3]:
for event in result.critique_iterations:
    draft = json.loads(event["proposal"])
    print(f"Round {event['iteration']}")
    for name in criteria:
        rule = REVIEW_CRITERIA[name]
        print(f"  {rule.label} [{rule.evidence_status(draft)}]: {draft[rule.field]}")
    print(f"  Critic: {event['feedback'] or 'Approved'}")
    for goal in event["revision_goals"]:
        print(f"    {goal}")
    print()

if result.terminated_reason == "max_iterations_reached":
    last = result.critique_iterations[-1]
    latest = json.loads(last["proposal"])
    open_criteria = [name for name in criteria if not REVIEW_CRITERIA[name].accepts(latest)]
    print(f"Round budget reached. Open criteria: {', '.join(open_criteria)}.")
    for goal in last["revision_goals"]:
        if goal.startswith("Change "):
            print(f"  Assigned for the next round, still unapplied: {goal}")

Round 1
  Kernel protection [missing]: Fixed clearance with no size adjustment during use.
  Cleaning access [unrecognized]: A hinged side panel exposes the separator and its collection tray.
  Critic: Kernel protection is missing. Specify how the operator can limit compression for different pod sizes. Cleaning access has an unrecognized description. A human reviewer can judge the existing feature.
    Change kernel protection: Adjustable roller gap and a compliant roller surface.

Round 2
  Kernel protection [accepted]: Adjustable roller gap and a compliant roller surface.
  Cleaning access [unrecognized]: A hinged side panel exposes the separator and its collection tray.
  Critic: Cleaning access has an unrecognized description. A human reviewer can judge the existing feature.
    Keep kernel protection: Adjustable roller gap and a compliant roller surface.
    Change cleaning access: Unclip the sieve and empty the removable collection tray.

Round 3
  Kernel protection [accepted]: A

## 5. Compare priorities with the same design budget

Give each team two rounds and one change between rounds. Both teams start with the three mechanisms you selected and fixed starting provisions for all three functions. Team A reviews kernel protection first. Team B reviews cleaning access first. Which features does each team carry into round two?

In [4]:
comparison_seed = ShellerConcept(feed, crack, separate)
with workshop_client() as client:
    for team, priority in [("A", "kernel protection"), ("B", "cleaning access")]:
        task = sheller_task(comparison_seed, priority, changes_per_revision=1)
        trial = sheller_pattern(client, max_iterations=2).run(task)
        last = trial.critique_iterations[-1]
        final_design = json.loads(last["proposal"])
        print(f"Team {team} prioritized {priority}.")
        print("  Kernel protection:", final_design["kernel_protection"])
        print("  Cleaning access:", final_design["cleaning_access"])
        pending = [goal for goal in last["revision_goals"] if goal.startswith("Change ")]
        for goal in pending:
            print("  Next change, still unapplied:", goal)
        print()

Team A prioritized kernel protection.
  Kernel protection: Adjustable roller gap and a compliant roller surface.
  Cleaning access: Fixed separation assembly with access from the outlet.
  Next change, still unapplied: Change cleaning access: Unclip the sieve and empty the removable collection tray.

Team B prioritized cleaning access.
  Kernel protection: Fixed clearance with no size adjustment during use.
  Cleaning access: Unclip the sieve and empty the removable collection tray.
  Next change, still unapplied: Change kernel protection: Adjustable roller gap and a compliant roller surface.



## Discuss

- Which first change would you choose for a grower who processes a mix of pod sizes? What information would change your choice?
- The critic approves a written provision. What would you measure on a prototype to judge kernel damage or ease of cleaning? Specify a comparison and an observation that would make you revise the design again.